In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "run.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from config.paths import load_paths
from data.prepare import prepare_array
PATHS = load_paths()


# MAE versus interval width

This notebook loads saved prediction intervals and plots point-forecast MAE across interval-width deciles.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.labelweight': 'bold',
    'axes.unicode_minus': False,
})

FIGSIZE = (5, 4)
LABEL_SIZE = 26
TICK_SIZE = 22
LEGEND_SIZE = 14

RESULT_ROOT = PATHS.results
FIGURE_DIR = RESULT_ROOT / 'figures'
SAVE_FIGURE = True
MAX_SAMPLES = 500_000
RANDOM_SEED = 2026
COMPONENT_ORDER = 'auto'  # 'auto', 'flow_cqr', or 'od_cqr'

# Each selection names an exact run directory under results/, plus its export.
# Example after an OD calibration run:
# SELECTIONS = [dict(run='my_experiment/<run-id>', pattern='*res.npy', label='PDR')]
SELECTIONS = []
DAILY_PROFILE_SELECTIONS = []
STEPS_PER_DAY = 24 * 4
DAY_START_OFFSET = 0  # Number of 15-min test steps before the first plotted midnight.


In [ ]:
def resolve_selection(selection):
    directory = RESULT_ROOT / selection['run']
    matches = sorted(directory.glob(selection.get('pattern', '*res*.npy')))
    if len(matches) != 1:
        raise ValueError(f'Expected one export in {directory}, found {len(matches)}; specify pattern.')
    return matches[0], selection['label']

@dataclass
class IntervalData:
    label: str
    point: np.ndarray
    lower: np.ndarray
    upper: np.ndarray
    target: np.ndarray

def _order_score(point, lower, upper):
    finite = np.isfinite(point) & np.isfinite(lower) & np.isfinite(upper)
    return np.mean((lower[finite] <= point[finite]) & (point[finite] <= upper[finite])) if finite.any() else -np.inf

def interval_component_indices(raw):
    probe = np.arange(min(10_000, int(np.prod(raw.shape[1:]))))
    candidates = {
        'flow_cqr': (raw[1].reshape(-1)[probe], raw[0].reshape(-1)[probe], raw[2].reshape(-1)[probe]),
        'od_cqr': (raw[0].reshape(-1)[probe], raw[1].reshape(-1)[probe], raw[2].reshape(-1)[probe]),
    }
    order = max(candidates, key=lambda key: _order_score(*candidates[key])) if COMPONENT_ORDER == 'auto' else COMPONENT_ORDER
    if order not in candidates:
        raise ValueError("COMPONENT_ORDER must be 'auto', 'flow_cqr', or 'od_cqr'.")
    return {'flow_cqr': (1, 0, 2), 'od_cqr': (0, 1, 2)}[order]

def load_interval_result(path, label):
    raw = np.load(path, mmap_mode='r')
    if raw.ndim < 2 or raw.shape[0] < 4:
        raise ValueError(f'{path.name} is not a four-component interval export; found shape {raw.shape}.')
    total = int(np.prod(raw.shape[1:]))
    take = total if MAX_SAMPLES is None else min(total, MAX_SAMPLES)
    rng = np.random.default_rng(RANDOM_SEED)
    indices = np.arange(total) if take == total else rng.choice(total, size=take, replace=False)

    point_index, lower_index, upper_index = interval_component_indices(raw)
    point = np.asarray(raw[point_index].reshape(-1)[indices], dtype=float)
    lower = np.asarray(raw[lower_index].reshape(-1)[indices], dtype=float)
    upper = np.asarray(raw[upper_index].reshape(-1)[indices], dtype=float)
    target = np.asarray(raw[3].reshape(-1)[indices], dtype=float)
    valid = np.isfinite(point) & np.isfinite(lower) & np.isfinite(upper) & np.isfinite(target) & (upper >= lower)
    return IntervalData(label, point[valid], lower[valid], upper[valid], target[valid])


In [ ]:
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
MARKERS = ['^', 'o', 's', 'D', 'v', 'P']

def method_style(label, index):
    name = label.lower()
    if 'zero' in name:
        return dict(color='#c51b7d', marker='D', zorder=8)
    if 'pdr' in name:
        return dict(color='#54278f', marker='^', zorder=7)
    return dict(color=COLORS[index % len(COLORS)], marker=MARKERS[index % len(MARKERS)], zorder=3 + index)

def width_decile_mae(data, bins=10):
    width = data.upper - data.lower
    edges = np.unique(np.quantile(width, np.linspace(0, 1, bins + 1)))
    group = np.clip(np.digitize(width, edges[1:-1], right=True), 0, len(edges) - 2)
    x = np.arange(len(edges) - 1) + 1
    mae = np.array([np.mean(np.abs(data.target[group == j] - data.point[group == j])) for j in range(len(x))])
    return x, mae

def plot_mae_vs_interval_width(datasets):
    fig, ax = plt.subplots(figsize=FIGSIZE)
    for index, data in enumerate(datasets):
        x, mae = width_decile_mae(data)
        ax.plot(x, mae, markerfacecolor='white', markeredgewidth=1.8, ms=8, lw=4,
                label=data.label, **method_style(data.label, index))
    ax.set_xlim(.7, 10.3)
    ax.set_xticks(np.arange(1, 11, 2))
    ax.set_xlabel('Interval (narrow → wide)', fontsize=LABEL_SIZE, labelpad=10)
    ax.set_ylabel('MAE', fontsize=LABEL_SIZE, labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=TICK_SIZE, direction='in', length=8, width=1.5)
    for spine in ax.spines.values():
        spine.set_linewidth(2)
    ax.grid(True, which='major', linestyle='--', linewidth=1, alpha=.4, zorder=0)

    legend = ax.legend(fontsize=LEGEND_SIZE, loc="lower right", frameon=True, framealpha=.9, edgecolor='gray',
                       handletextpad=.4, borderpad=.35)
    legend.get_frame().set_linewidth(1.5)
    fig.tight_layout()
    return fig

datasets = [load_interval_result(*resolve_selection(selection)) for selection in SELECTIONS]
fig = plot_mae_vs_interval_width(datasets)
if SAVE_FIGURE:
    print(123)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / 'mae_vs_interval_width.pdf', bbox_inches='tight')
plt.show()


# Average daily prediction profile

Each curve averages all OD pairs and all complete test days at the same 15-minute slot.

In [ ]:
def load_daily_profile(path, label):
    raw = np.load(path, mmap_mode='r')
    if raw.ndim < 3 or raw.shape[0] < 4 or raw.shape[2] != 1:
        raise ValueError(f'{path.name} must contain one forecast horizon per test step; found shape {raw.shape}.')
    point_index, lower_index, upper_index = interval_component_indices(raw)
    # Average all OD pairs, retaining the chronological test-step axis.
    reduce_axes = tuple(range(1, raw.ndim - 1))
    point = np.asarray(raw[point_index].mean(axis=reduce_axes), dtype=float)
    lower = np.asarray(raw[lower_index].mean(axis=reduce_axes), dtype=float)
    upper = np.asarray(raw[upper_index].mean(axis=reduce_axes), dtype=float)
    target = np.asarray(raw[3].mean(axis=reduce_axes), dtype=float)
    start = DAY_START_OFFSET % STEPS_PER_DAY
    usable = ((len(point) - start) // STEPS_PER_DAY) * STEPS_PER_DAY
    if usable == 0:
        raise ValueError('The exported test sequence does not contain one complete day after DAY_START_OFFSET.')
    def daily_mean(values):
        return values[start:start + usable].reshape(-1, STEPS_PER_DAY).mean(axis=0)
    return dict(label=label, point=daily_mean(point), lower=daily_mean(lower),
                upper=daily_mean(upper), target=daily_mean(target))

def plot_average_daily_profile(profiles):
    fig, ax = plt.subplots(figsize=FIGSIZE)
    x = np.arange(STEPS_PER_DAY)
    reference = profiles[0]['target']
    for profile in profiles[1:]:
        if not np.allclose(reference, profile['target'], equal_nan=True):
            raise ValueError('The selected methods do not share the same ground-truth test sequence.')
    ax.plot(x, reference, color='black', lw=4.5, label='Ground Truth', zorder=5)
    for profile, selection in zip(profiles, DAILY_PROFILE_SELECTIONS):
        if selection['interval']:
            ax.fill_between(x, profile['lower'], profile['upper'], color=selection['color'], alpha=.18, linewidth=0,
                            zorder=1)
        ax.plot(x, profile['point'], color=selection['color'], linestyle=selection['linestyle'], lw=3.2,
                label=profile['label'], zorder=4)
    ticks = np.arange(0, STEPS_PER_DAY + 1, 16)
    ax.set_xticks(ticks, [f'{hour:02d}:00' for hour in range(0, 25, 4)])
    ax.set_xlim(0, STEPS_PER_DAY - 1)
    ax.set_xlabel('Time of day', fontsize=LABEL_SIZE, labelpad=10)
    ax.set_ylabel('Mean OD demand', fontsize=LABEL_SIZE, labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=TICK_SIZE, direction='in', length=8, width=1.5)
    for spine in ax.spines.values():
        spine.set_linewidth(2)
    ax.grid(True, which='major', linestyle='--', linewidth=1, alpha=.4, zorder=0)
    legend = ax.legend(fontsize=LEGEND_SIZE, ncol=2, loc='lower left', bbox_to_anchor=(0, 1.01),
                       frameon=True, framealpha=.9, edgecolor='gray', columnspacing=.9, handletextpad=.4, borderpad=.35)
    legend.get_frame().set_linewidth(1.5)
    fig.tight_layout()
    return fig

daily_profile_files = [resolve_selection(selection) for selection in DAILY_PROFILE_SELECTIONS]
daily_profiles = [load_daily_profile(path, label) for path, label in daily_profile_files]
fig_daily = plot_average_daily_profile(daily_profiles)
if SAVE_FIGURE:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig_daily.savefig(FIGURE_DIR / 'average_daily_prediction_profile.pdf', bbox_inches='tight')
plt.show()
